<a href="https://colab.research.google.com/github/aryam-al/aryam/blob/master/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Fri Sep 25 20:36:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U bitsandbytes


In [ ]:
import json
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# BioMistral-7B is ungated (Apache 2.0) — no HF token or login needed.

# --- 1. Load Dataset & Prepare 100 Test Cases ---
df = pd.read_csv('Autoimmune_Dataset_Junior_Balanced_clean.csv')

target_col = 'Disease' if 'Disease' in df.columns else 'Label'
X = df.drop(columns=[target_col])
y = df[target_col]

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

test_100 = X_test.head(100).copy()
true_labels = y_test.head(100).tolist()

formatted_patients = []
for idx, (original_idx, row) in enumerate(test_100.iterrows()):
    antibodies = [c for c in test_100.columns if c not in ['Age', 'Gender', 'ESR', 'CRP', 'C3', 'C4']]
    positive_ab = [ab for ab in antibodies if row[ab] == 1]

    patient_dict = {
        "patient_id": idx + 1,
        "Age": int(row['Age']),
        "Gender": row['Gender'],
        "ESR": float(row['ESR']),
        "CRP": float(row['CRP']),
        "C3": float(row['C3']),
        "C4": float(row['C4']),
        "Positive_Antibodies": positive_ab
    }
    formatted_patients.append(patient_dict)

# Map patient_id -> true label, built directly from the source data.
true_label_by_id = {idx + 1: label for idx, label in enumerate(true_labels)}

print(f"Successfully formatted {len(formatted_patients)} patients into test JSON.")

# --- 2. Prompts Setup ---
system_prompt = """You are an expert rheumatologist acting as a diagnostic classifier. Analyze clinical biomarker profiles.
You must assign EXACTLY ONE primary diagnosis per patient from this exact list:
["Normal", "Rheumatoid Arthritis", "Psoriatic Arthritis", "Systemic Lupus Erythematosus", "Multiple sclerosis", "Ankylosing Spondylitis", "Celiac disease", "Reactive Arthritis", "Crohn's disease", "Sjögren's Syndrome"]

Return ONLY a valid JSON array of objects with keys: "patient_id", "predicted_diagnosis", "rationale"."""

one_shot_example = """EXAMPLE CASE:
Input: {"patient_id": 0, "Age": 50, "Gender": "Female", "ESR": 32.0, "CRP": 29.4, "C3": 205.0, "C4": 16.0, "Positive_Antibodies": ["RF", "Anti_CCP"]}
Output: {"patient_id": 0, "predicted_diagnosis": "Rheumatoid Arthritis", "rationale": "Elevated ESR/CRP paired with positive RF and Anti_CCP point to Rheumatoid Arthritis."}"""

# --- 3. Load BioMistral-7B (medical-domain, ungated, Apache 2.0) ---
model_id = "BioMistral/BioMistral-7B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {model_id} onto GPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    max_memory={0: "14GiB", "cpu": "0GiB"}
)

# NOTE: BioMistral's native context window is only 2048 tokens (much
# smaller than Qwen/Llama's 128K). If a batch of 10 patients plus the
# one-shot example ever approaches that limit, reduce batch_size below.

# --- 4. Run Batch Inference ---
batch_size = 5  # smaller than Qwen/Llama runs, since BioMistral's context window is only 2048 tokens
all_predictions = []
failed_batches = []
failed_batch_raw_output = {}

print("\nStarting batch evaluation with BioMistral-7B across 100 cases...\n")

for i in range(0, len(formatted_patients), batch_size):
    batch_num = i // batch_size + 1
    batch_patients = formatted_patients[i:i + batch_size]
    user_prompt = f"{one_shot_example}\n\nPatients to classify:\n{json.dumps(batch_patients)}"

    # BioMistral's chat template (inherited from Mistral-7B-Instruct-v0.1) does
    # NOT support a separate "system" role — only alternating user/assistant.
    # So we fold the system instructions into the single user turn instead.
    combined_prompt = f"{system_prompt}\n\n{user_prompt}"
    messages = [
        {"role": "user", "content": combined_prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1500).to("cuda")
    input_len = inputs["input_ids"].shape[-1]
    if input_len >= 1500:
        print(f"WARNING: batch {batch_num} prompt was truncated at 1500 tokens (context limit 2048) — reduce batch_size further if this happens often.")

    outputs = model.generate(**inputs, max_new_tokens=500, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    # Try strict JSON array parsing first
    try:
        json_str = response_text[response_text.find('['):response_text.rfind(']')+1]
        batch_preds = json.loads(json_str)
        all_predictions.extend(batch_preds)
        print(f"Processed batch {batch_num}/10 successfully ({len(batch_preds)} predictions).")
        continue
    except Exception as e:
        pass

    # Fallback: Llama sometimes prints extra text after the JSON array
    # ("Extra data" errors). Try to salvage by parsing objects one at a time.
    try:
        import re
        objects = re.findall(r'\{[^{}]*\}', response_text)
        batch_preds = [json.loads(o) for o in objects]
        if batch_preds:
            all_predictions.extend(batch_preds)
            print(f"Processed batch {batch_num}/10 via fallback parsing ({len(batch_preds)} predictions).")
            continue
    except Exception:
        pass

    failed_batches.append(batch_num)
    failed_batch_raw_output[batch_num] = response_text[:800]
    print(f"Parsing error in batch {batch_num} (both strict and fallback parsing failed).")

# --- 5. Match predictions to true labels by patient_id (NOT by list position) ---
pred_by_id = {}
for p in all_predictions:
    pid = p.get('patient_id')
    if pid is not None:
        pred_by_id[pid] = p

correct = 0
missing = 0
mismatched_examples = []
prediction_counts = {}

for pid, actual_raw in true_label_by_id.items():
    actual = str(actual_raw).strip().lower()
    pred = pred_by_id.get(pid)

    if pred is None:
        missing += 1
        continue

    predicted_raw = pred.get('predicted_diagnosis', '')
    predicted = str(predicted_raw).strip().lower()
    prediction_counts[predicted_raw] = prediction_counts.get(predicted_raw, 0) + 1

    if predicted == actual:
        correct += 1
    elif len(mismatched_examples) < 10:
        mismatched_examples.append((pid, actual_raw, predicted_raw))

evaluated = len(true_label_by_id) - missing
accuracy_of_evaluated = (correct / evaluated * 100) if evaluated else 0
accuracy_of_all_100 = (correct / len(true_label_by_id) * 100) if true_label_by_id else 0

naive_baseline_correct = sum(
    1 for label in true_label_by_id.values() if str(label).strip().lower() == 'normal'
)
naive_baseline_pct = naive_baseline_correct / len(true_label_by_id) * 100

print("\n" + "="*50)
print("RESULTS - BioMistral-7B")
print("="*50)
print(f"Failed batches (parsing errors): {failed_batches if failed_batches else 'none'}")
print(f"Predictions received: {len(all_predictions)} / 100")
print(f"Missing predictions (no patient_id match): {missing}")
print(f"Correct: {correct}")
print(f"Accuracy over evaluated cases ({evaluated} cases): {accuracy_of_evaluated:.2f}%")
print(f"Accuracy over all 100 cases (missing = wrong): {accuracy_of_all_100:.2f}%")
print(f"Naive 'always predict Normal' baseline: {naive_baseline_pct:.2f}%")
print("\nModel's predicted-diagnosis distribution:")
for label, count in sorted(prediction_counts.items(), key=lambda x: -x[1]):
    print(f"  {label}: {count}")
print("\nSample mismatches (patient_id, true, predicted):")
for pid, actual_raw, predicted_raw in mismatched_examples:
    print(f"  #{pid}: true='{actual_raw}' | predicted='{predicted_raw}'")
if failed_batches:
    print("\nRaw output for failed batches (for debugging):")
    for bn, raw in failed_batch_raw_output.items():
        print(f"--- Batch {bn} ---\n{raw}\n")
print("="*50)


Successfully formatted 100 patients into test JSON.
Loading BioMistral/BioMistral-7B onto GPU...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Starting batch evaluation with BioMistral-7B across 100 cases...

Processed batch 1/10 successfully (5 predictions).
Processed batch 2/10 successfully (5 predictions).
Processed batch 3/10 successfully (5 predictions).
Processed batch 4/10 successfully (5 predictions).
Processed batch 5/10 successfully (5 predictions).
Processed batch 6/10 successfully (5 predictions).
Processed batch 7/10 successfully (5 predictions).
Processed batch 8/10 successfully (5 predictions).
Processed batch 9/10 successfully (5 predictions).
Processed batch 10/10 successfully (5 predictions).
Processed batch 11/10 successfully (5 predictions).
Processed batch 12/10 successfully (5 predictions).
Processed batch 13/10 successfully (5 predictions).
Processed batch 14/10 successfully (5 predictions).
Processed batch 15/10 successfully (5 predictions).
Processed batch 16/10 successfully (5 predictions).
Processed batch 17/10 successfully (5 predictions).
Processed batch 18/10 successfully (5 predictions).
Proces